# 03b — Train `tomato_leaf_disease_v1` với augmentation màu/ánh sáng tăng cường

**Bối cảnh:** `03_train_leaf_baseline.ipynb` (sau khi loại 62 ảnh gắn nhãn Healthy sai) đạt Precision 0,918 / Recall 0,872 / mAP@0.5 0,937 trên `test`, nhưng còn 2 điểm yếu:
1. `leaf_mold` là class yếu nhất trong 4 bệnh (Recall 0,774, AP50 0,876 — thấp hơn hẳn 3 class còn lại đang ở 0,93–0,97).
2. Tỷ lệ báo động giả trên ảnh lá khỏe vẫn còn 19,7% (12/61) — vượt ngưỡng cảnh báo 20% khá sát.

**Giả thuyết:** khác với nhánh độ chín (nơi đã đo được domain shift thật giữa train và test), nhánh lá không có domain gap đã biết — `test` cùng phân bố với `train/val`. Nhưng dữ liệu Roboflow đã tự augment sẵn (rotate/flip/exposure) nên model có thể đang học hơi "cứng" theo đúng tông màu/ánh sáng của bộ augmentation gốc đó. Tăng thêm augmentation màu/ánh sáng khi train — đúng cách đã chứng minh hiệu quả bất ngờ ở `02c_train_ripeness_augmented.ipynb` (giảm cả nhầm lẫn lẫn báo động giả dù ban đầu chỉ nhắm vào màu sắc) — có thể giúp model bớt nhạy cảm với texture/màu cụ thể, giảm báo động giả trên lá khỏe.

**Đây là thử nghiệm, không phải chắc chắn có ích** — notebook đo đầy đủ cả 2 khía cạnh (metrics tổng thể + tỷ lệ báo động giả) và so trực tiếp với baseline để biết có thật sự tốt hơn không, tránh suy đoán.

## Trước khi chạy
1. **Add Input** → output đã Save Version mới nhất của `01_build_tomato_leaf_disease_v1.ipynb` (`tomato_leaf_disease_v1`, đã loại 62 ảnh gắn nhãn sai).
2. Accelerator: **GPU** (P100/T4x2) bắt buộc. Internet: ON.

## Cấu hình
Giống hệt `03_train_leaf_baseline.ipynb` (YOLOv8n, imgsz 640, epochs 100, batch 16, patience 20, seed 42) — chỉ khác augmentation màu, dùng đúng bộ tham số đã hiệu quả ở nhánh độ chín:

| Tham số | Mặc định ultralytics | Notebook này |
|---|---|---|
| `hsv_h` (hue) | 0.015 | 0.02 |
| `hsv_s` (saturation) | 0.7 | 0.9 |
| `hsv_v` (brightness) | 0.4 | 0.6 |

In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "ultralytics"], check=False)

import torch

print("CUDA khả dụng:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "Không có GPU. Vào Settings (panel phải) -> Accelerator -> chọn GPU T4 x2 hoặc P100, "
        "rồi chạy lại từ đầu."
    )

### Bước 1 — Tự nhận diện dataset + vá `data.yaml`

In [ ]:
from pathlib import Path

import yaml

TARGET_CLASSES = ["leaf_early_blight", "leaf_late_blight", "leaf_mold", "leaf_septoria_spot"]


def find_dataset_root():
    for yf in Path("/kaggle/input").rglob("data.yaml"):
        try:
            y = yaml.safe_load(yf.read_text(encoding="utf-8"))
        except Exception:
            continue
        names = y.get("names") if isinstance(y, dict) else None
        if isinstance(names, dict):
            names = [names[k] for k in sorted(names)]
        if names == TARGET_CLASSES:
            return yf.parent
    return None


DATASET_ROOT = find_dataset_root()
if DATASET_ROOT is None:
    print("Các thư mục cấp 1 trong /kaggle/input:")
    for p in sorted(Path("/kaggle/input").glob("*")):
        print(" -", p)
    raise FileNotFoundError(
        "Không tìm thấy data.yaml khớp 4 class của tomato_leaf_disease_v1 trong /kaggle/input."
    )

DATA_YAML = DATASET_ROOT / "data.yaml"
print("DATASET_ROOT:", DATASET_ROOT)

WORKING = Path("/kaggle/working")
WORKING.mkdir(parents=True, exist_ok=True)

orig_yaml = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
fixed_yaml = {
    "path": str(DATASET_ROOT),
    "train": orig_yaml.get("train", "train/images"),
    "val": orig_yaml.get("val", "val/images"),
    "test": orig_yaml.get("test", "test/images"),
    "names": orig_yaml["names"],
}
FIXED_DATA_YAML = WORKING / "data_leaf.yaml"
with open(FIXED_DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(fixed_yaml, f, allow_unicode=True, sort_keys=False)
print("data.yaml đã vá:", FIXED_DATA_YAML)

for split_key in ["train", "val", "test"]:
    img_dir = DATASET_ROOT / fixed_yaml[split_key]
    n = len(list(img_dir.glob("*"))) if img_dir.exists() else 0
    print(f"  {split_key:6s}: {n} ảnh")
    if n == 0:
        raise FileNotFoundError(f"{split_key} rỗng tại {img_dir}.")

### Bước 2 — Train YOLOv8n với augmentation màu tăng cường
Ảnh negative (lá khỏe) được YOLO hỗ trợ natively — không cần xử lý gì thêm.

In [ ]:
from ultralytics import YOLO

RUN_NAME = "tomato_leaf_disease_v1_yolov8n_augmented"
RUNS_DIR = WORKING / "runs"

model = YOLO("yolov8n.pt")

train_results = model.train(
    data=str(FIXED_DATA_YAML),
    imgsz=640,
    epochs=100,
    batch=16,
    patience=20,
    seed=42,
    pretrained=True,
    hsv_h=0.02,
    hsv_s=0.9,
    hsv_v=0.6,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
)

BEST_PT = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
print("\nbest.pt:", BEST_PT, "tồn tại:", BEST_PT.exists())

### Bước 3 — Đánh giá trên `test` + so trực tiếp với baseline
Số liệu baseline (từ `03_train_leaf_baseline.ipynb`, sau khi đã loại 62 ảnh gắn nhãn sai) đã lưu cứng để so sánh vì đây là notebook độc lập, không đọc lại được run trước trên Kaggle.

In [ ]:
import pandas as pd

best_model = YOLO(str(BEST_PT))

test_metrics = best_model.val(
    data=str(FIXED_DATA_YAML), split="test", imgsz=640,
    project=str(RUNS_DIR), name=f"{RUN_NAME}_eval_test", exist_ok=True,
)

BASELINE_OVERALL = {"precision": 0.9179, "recall": 0.8717, "map50": 0.9368, "map50_95": 0.8232}
BASELINE_PER_CLASS = {
    "leaf_early_blight": {"precision": 0.9218, "recall": 0.9204, "ap50": 0.9701, "ap50_95": 0.9033},
    "leaf_late_blight":  {"precision": 0.9247, "recall": 0.9522, "ap50": 0.9724, "ap50_95": 0.8738},
    "leaf_mold":         {"precision": 0.8983, "recall": 0.7736, "ap50": 0.8756, "ap50_95": 0.7310},
    "leaf_septoria_spot":{"precision": 0.9268, "recall": 0.8406, "ap50": 0.9293, "ap50_95": 0.7846},
}

print("== test — augmented vs baseline ==")
print(f"{'metric':12s} {'baseline':>10s} {'augmented':>10s} {'delta':>8s}")
for k in ["precision", "recall", "map50", "map50_95"]:
    aug_val = {"precision": test_metrics.box.mp, "recall": test_metrics.box.mr,
               "map50": test_metrics.box.map50, "map50_95": test_metrics.box.map}[k]
    print(f"{k:12s} {BASELINE_OVERALL[k]:10.4f} {float(aug_val):10.4f} {float(aug_val) - BASELINE_OVERALL[k]:+8.4f}")

print("\nTheo từng class (AP50, baseline -> augmented):")
per_class_rows = []
for i, cname in enumerate(TARGET_CLASSES):
    aug_p, aug_r = float(test_metrics.box.p[i]), float(test_metrics.box.r[i])
    aug_ap50, aug_ap5095 = float(test_metrics.box.ap50[i]), float(test_metrics.box.ap[i])
    base = BASELINE_PER_CLASS[cname]
    print(f"  {cname:20s} AP50 {base['ap50']:.4f} -> {aug_ap50:.4f} ({aug_ap50 - base['ap50']:+.4f})  "
          f"R {base['recall']:.4f} -> {aug_r:.4f} ({aug_r - base['recall']:+.4f})")
    per_class_rows.append({"class": cname, "precision": round(aug_p, 4), "recall": round(aug_r, 4),
                            "ap50": round(aug_ap50, 4), "ap50_95": round(aug_ap5095, 4),
                            "baseline_ap50": base["ap50"], "delta_ap50": round(aug_ap50 - base["ap50"], 4)})

delta_map50 = float(test_metrics.box.map50) - BASELINE_OVERALL["map50"]
if delta_map50 > 0.01:
    print(f"\n[KẾT QUẢ] mAP@0.5 tổng thể tăng {delta_map50:+.4f} — augmentation có giúp ích.")
elif delta_map50 < -0.01:
    print(f"\n[KẾT QUẢ] mAP@0.5 tổng thể giảm {delta_map50:+.4f} — augmentation mạnh hơn không giúp cho nhánh "
          "này, có thể do dữ liệu Roboflow đã tự augment sẵn nên tăng thêm chỉ gây nhiễu. Nên quay lại baseline.")
else:
    print(f"\n[KẾT QUẢ] mAP@0.5 gần như không đổi ({delta_map50:+.4f}) — xem thêm tỷ lệ báo động giả ở Bước 4 "
          "trước khi kết luận có nên đổi model hay không.")

### Bước 4 — Tỷ lệ báo động giả trên ảnh negative (so với baseline 19,7%)
Đây là chỉ số quan trọng nhất để đánh giá giả thuyết của notebook này — nếu giảm rõ so với 19,7%, augmentation màu thực sự giúp giảm nhạy cảm với texture/màu.

In [ ]:
BASELINE_FP_RATE = 0.197  # 12/61 o 03_train_leaf_baseline.ipynb (sau khi da loai 62 anh gan nhan sai)

test_images_dir = DATASET_ROOT / fixed_yaml["test"]
test_labels_dir = test_images_dir.parent / "labels"

negative_images = []
for img_path in sorted(test_images_dir.glob("*")):
    label_path = test_labels_dir / (img_path.stem + ".txt")
    lines = [l for l in label_path.read_text().splitlines() if l.strip()] if label_path.exists() else []
    if not lines:
        negative_images.append(img_path)

print(f"Số ảnh negative trong test: {len(negative_images)} / {len(list(test_images_dir.glob('*')))}")

fp_rate = None
if negative_images:
    neg_results = best_model.predict(
        source=[str(p) for p in negative_images], imgsz=640, conf=0.25, verbose=False
    )
    n_false_triggered = sum(1 for r in neg_results if len(r.boxes) > 0)
    fp_rate = n_false_triggered / len(negative_images)
    print(f"Ảnh negative bị báo nhầm có bệnh (>=1 box dự đoán): {n_false_triggered} / {len(negative_images)} "
          f"({100 * fp_rate:.1f}%)")
    print(f"So với baseline: {100 * BASELINE_FP_RATE:.1f}% -> {100 * fp_rate:.1f}% "
          f"({100 * (fp_rate - BASELINE_FP_RATE):+.1f} điểm %)")
    if fp_rate < BASELINE_FP_RATE - 0.03:
        print("[KẾT QUẢ] Giảm rõ rệt — augmentation màu giúp thật với báo động giả.")
    elif fp_rate > BASELINE_FP_RATE + 0.03:
        print("[KẾT QUẢ] Tăng lên — augmentation màu không giúp, thậm chí phản tác dụng cho chỉ số này.")
    else:
        print("[KẾT QUẢ] Gần như không đổi so với baseline.")
else:
    print("Không có ảnh negative nào trong test -> bỏ qua bước này.")

### Bước 5 — Lưu model + báo cáo so sánh

In [ ]:
import shutil

MODELS_DIR = WORKING / "models"
REPORTS_DIR = WORKING / "reports" / RUN_NAME
MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy2(BEST_PT, MODELS_DIR / "tomato_leaf_disease_yolov8n_augmented.pt")

pd.DataFrame(per_class_rows).to_csv(REPORTS_DIR / "metrics_per_class_vs_baseline.csv", index=False)

summary_row = {
    "precision": round(float(test_metrics.box.mp), 4), "recall": round(float(test_metrics.box.mr), 4),
    "map50": round(float(test_metrics.box.map50), 4), "map50_95": round(float(test_metrics.box.map), 4),
    "baseline_map50": BASELINE_OVERALL["map50"], "delta_map50": round(delta_map50, 4),
    "negative_false_positive_rate": round(fp_rate, 4) if fp_rate is not None else None,
    "baseline_fp_rate": BASELINE_FP_RATE,
}
pd.DataFrame([summary_row]).to_csv(REPORTS_DIR / "metrics_summary_vs_baseline.csv", index=False)

train_config = {
    "model": "yolov8n.pt", "imgsz": 640, "epochs": 100, "batch": 16, "patience": 20, "seed": 42,
    "pretrained": True, "hsv_h": 0.02, "hsv_s": 0.9, "hsv_v": 0.6,
    "note": "Tang cuong augmentation mau/anh sang so voi baseline, thu nghiem giam bao dong gia + cai thien leaf_mold.",
}
with open(REPORTS_DIR / "train_config.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(train_config, f, allow_unicode=True, sort_keys=False)

print("Đã lưu:")
print(" -", MODELS_DIR / "tomato_leaf_disease_yolov8n_augmented.pt")
print(" -", REPORTS_DIR / "metrics_per_class_vs_baseline.csv")
print(" -", REPORTS_DIR / "metrics_summary_vs_baseline.csv")
print(" -", REPORTS_DIR / "train_config.yaml")

### Bước 6 — Xác nhận trực quan: dự đoán thật trên ảnh mẫu
Xem cả ảnh dương (có bệnh) lẫn ảnh negative (lá khỏe) — không chỉ tin số liệu.

In [ ]:
import random

import matplotlib.pyplot as plt

random.seed(42)


def show_predictions(img_paths, n=6, title=""):
    img_paths = list(img_paths)
    if not img_paths:
        print(f"[{title}] Không có ảnh để hiển thị.")
        return
    sample = random.sample(img_paths, min(n, len(img_paths)))
    results = best_model.predict(source=[str(p) for p in sample], imgsz=640, conf=0.25, verbose=False)

    fig, axes = plt.subplots(1, len(sample), figsize=(3.6 * len(sample), 3.6))
    axes = [axes] if len(sample) == 1 else axes
    for ax, res, img_path in zip(axes, results, sample):
        annotated = res.plot()[:, :, ::-1]  # BGR -> RGB
        ax.imshow(annotated)
        ax.set_title(img_path.name, fontsize=6)
        ax.axis("off")
    fig.suptitle(title, fontsize=10)
    plt.tight_layout()
    save_path = REPORTS_DIR / f"predictions_{title.replace(' ', '_')}.png"
    plt.savefig(save_path, dpi=80, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Đã lưu:", save_path)


positive_images = [p for p in sorted(test_images_dir.glob("*")) if p not in set(negative_images)]
show_predictions(positive_images, title="test_positive")
show_predictions(negative_images, title="test_negative_healthy")

## Kết quả & cách đọc
- So mAP@0.5 tổng thể và đặc biệt AP50/Recall của `leaf_mold` (Bước 3) với baseline (0,8756 / 0,7736) — đây là class notebook này nhắm tới cải thiện nhiều nhất.
- So tỷ lệ báo động giả (Bước 4) với baseline 19,7% — chỉ số quan trọng nhất để kết luận giả thuyết đúng hay sai.
- Nếu **cả hai đều tốt hơn hoặc bằng** baseline mà không đánh đổi các class khác: dùng `models/tomato_leaf_disease_yolov8n_augmented.pt` thay baseline.
- Nếu chỉ 1 trong 2 tốt hơn, hoặc cả hai tệ hơn: giữ nguyên baseline gốc — augmentation không phải lúc nào cũng có ích, nhánh lá có thể đã ở gần giới hạn của baseline nano + dữ liệu hiện có (theo đúng mục 4 trong README: cần dữ liệu tốt hơn, không phải chỉ đổi tham số train).

## Bước tiếp theo
- Cập nhật `ai/README.md` với kết quả thật (thay placeholder này).
- Nếu augmented tốt hơn: coi đây là baseline chính thức mới cho nhánh lá, tương tự cách đã làm với `02c_train_ripeness_augmented.ipynb` ở nhánh độ chín.